# quantbench_regime_cadence_screen_v1

Executable diagnostic for the simplified non-RL SOXL path: qlib-backed supervised QuantBench-style models, existing SOXL factor packs, and a regime-aware trade-cadence overlay.

This card is intentionally a screen, not a promotion. It checks whether dynamic trade frequency can be expressed and measured inside the supervised backtest path before any broader reinforcement-learning cleanup is attempted.

## Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `quantbench_paper_soxl_v2` supervised no-cadence rows.
- **Role**: diagnose.
- **Target window**: `test_2022`, plus `test_2020`, `test_2023_2024`, `latest_2025_2026`, and `ytd_2026` slices.
- **Accepted trade-off**: lower trading frequency or turnover in stress/range regimes without claiming promotion.
- **Changed layer**: execution cadence and model-selection evidence only.
- **External references**: QuantBench paper model families, qlib supervised DatasetH surface, and Kaggle/Jane Street validation discipline: time-ordered local validation, leakage avoidance, and no promotion from a single leaderboard-like proxy.
- **Closest success case**: `quantbench_paper_soxl_v2` found at least one eligible supervised ensemble after paper-family expansion.
- **Closest failure case**: `quantbench_paper_soxl_v2` found that single supervised rows generally failed the 20% MaxDD gate.
- **Why this is not a duplicate**: this screen changes execution cadence only and compares cadence rows against matched no-cadence rows; it does not add another renamed profile or RL penalty knob.
- **Comparison-table row key**: `profile` in `supervised_candidate_screen.csv` and `cadence_profile` in `cadence_pair_metrics.csv`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")


ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "quantbench_regime_cadence_screen_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

from src.soxl_quant.app.quantbench_screen import QuantBenchGate
from src.soxl_quant.app.quantbench_supervised import (
    build_quantbench_supervised_report,
    quantbench_risk_overlay_configs,
    quantbench_trade_cadence_configs,
)
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

print(f"root={ROOT.relative_to(ROOT)}")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model_selection/card/quantbench_regime_cadence_screen_v1


## Data And Feature Surface

This diagnostic uses the repo-owned qlib dataset path and existing SOXL feature packs. It does **not** yet replace the feature surface with stock qlib `Alpha158`; that remains a later architecture simplification step.

In [2]:
feature_packs = (
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation",
)

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=feature_packs, profile_name="quantbench_regime_cadence_screen_v1")
features = dataset.handler.feature_frame
close = dataset.handler.price_frame["price"]

if isinstance(features.index, pd.MultiIndex) and "instrument" in features.index.names:
    feature_index = pd.to_datetime(features.index.get_level_values("datetime"))
else:
    feature_index = pd.to_datetime(features.index)

model_test_end = str(pd.Series(feature_index).max().date())
try:
    qlib_path = str(QLIB_DATA_PATH.relative_to(ROOT))
except ValueError:
    qlib_path = str(QLIB_DATA_PATH)
DATA_CONTEXT = {
    "qlib_data_path": qlib_path,
    "feature_rows": int(len(features)),
    "feature_columns": int(features.shape[1]),
    "feature_start": str(pd.Series(feature_index).min().date()),
    "feature_end": model_test_end,
    "feature_packs": list(feature_packs),
}
print(json.dumps(DATA_CONTEXT, indent=2, sort_keys=True))

[1757:MainThread](2026-06-17 21:52:17,300) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[1757:MainThread](2026-06-17 21:52:17,737) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[1757:MainThread](2026-06-17 21:52:17,737) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/quantbench-regime-cadence-v1/data/qlib/soxl')}


[1757:MainThread](2026-06-17 21:52:19,400) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[1757:MainThread](2026-06-17 21:52:19,400) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[1757:MainThread](2026-06-17 21:52:19,401) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


{
  "feature_columns": 113,
  "feature_end": "2026-06-15",
  "feature_packs": [
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation"
  ],
  "feature_rows": 3838,
  "feature_start": "2011-03-11",
  "qlib_data_path": "data/qlib/soxl"
}


## Focused Cadence Screen

The model set is deliberately narrow: two simple tabular baselines, one tree baseline, one lagged sequence baseline, one lightweight temporal convolution, and one graph-style ridge proxy. The goal is to measure the execution-cadence layer, not to rerun the full 44-spec QuantBench matrix.

In [3]:
model_names = (
    "logistic_clf",
    "ridge_mse",
    "random_forest_clf",
    "sequence_lag_ridge_mse",
    "tcn_mse",
    "market_graph_mse",
)

windows = {
    "test_2020": ("2020-01-01", "2020-12-31"),
    "test_2022": ("2022-01-01", "2022-12-31"),
    "test_2023_2024": ("2023-01-01", "2024-12-31"),
    "latest_2025_2026": ("2025-01-01", model_test_end),
    "ytd_2026": ("2026-01-01", model_test_end),
}

risk_configs = tuple(quantbench_risk_overlay_configs())
cadence_configs = tuple(quantbench_trade_cadence_configs())

report = build_quantbench_supervised_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    model_names=model_names,
    feature_packs=feature_packs,
    train_start="2010-03-11",
    train_end="2018-12-31",
    test_start="2020-01-01",
    test_end=model_test_end,
    profile_prefix="quantbench_regime_cadence_screen_v1",
    gate=QuantBenchGate(max_drawdown_pct=20.0),
    min_position=0.10,
    max_position=0.90,
    cost=0.0008,
    risk_configs=risk_configs,
    cadence_configs=cadence_configs,
    windows=windows,
    rolling_enabled=False,
)

print(json.dumps({
    "records": report["screen"]["records"],
    "eligible_count": report["screen"]["eligible_count"],
    "trade_cadences": report["data"]["trade_cadences"],
    "risk_overlays": report["data"]["risk_overlays"],
}, indent=2, sort_keys=True))

{
  "eligible_count": 0,
  "records": 60,
  "risk_overlays": [
    "macro_budget_defensive_v1",
    "macro_budget_lowdd_v1",
    "macro_budget_bull_guard_v1",
    "macro_budget_reentry_v1"
  ],
  "trade_cadences": [
    "regime_cadence_fast_bull_slow_stress_v1"
  ]
}


## Matched Cadence Deltas

Each cadence row is matched to the same model and risk overlay without the cadence suffix. Negative `delta_*_trades` and `delta_*_turnover` values mean the regime cadence reduced trading frequency or turnover versus the matched no-cadence row.

In [4]:
payloads_path = CARD / "supervised_payloads.json"
payloads = json.loads(payloads_path.read_text(encoding="utf-8"))
payload_by_profile = {payload["profile"]: payload for payload in payloads}
cadence_name = cadence_configs[0].name


def metric_at(payload: dict[str, Any], slice_name: str | None, metric: str) -> float:
    source = payload.get("test_proxy", {}) if slice_name is None else payload.get("slices", {}).get(slice_name, {})
    value = source.get(metric)
    return float(value) if value is not None and value == value else float("nan")


def safe_delta(left: float, right: float) -> float:
    if not np.isfinite(left) or not np.isfinite(right):
        return float("nan")
    return float(left - right)

rows: list[dict[str, Any]] = []
for cadence_payload in payloads:
    profile = str(cadence_payload["profile"])
    if f"_{cadence_name}" not in profile:
        continue
    base_profile = profile.replace(f"_{cadence_name}", "")
    base_payload = payload_by_profile.get(base_profile)
    if base_payload is None:
        continue
    row: dict[str, Any] = {
        "cadence_profile": profile,
        "base_profile": base_profile,
        "model_name": cadence_payload.get("model_name"),
        "paper_family": cadence_payload.get("paper_family"),
        "risk_overlay": (cadence_payload.get("risk_overlay") or {}).get("name", "none"),
        "trade_cadence": cadence_name,
    }
    for label, slice_name in (("proxy", None), ("test_2022", "test_2022"), ("latest_2025_2026", "latest_2025_2026")):
        for metric in ("return", "max_dd", "turnover", "trades", "cadence_blocked_days"):
            cadence_value = metric_at(cadence_payload, slice_name, metric)
            base_value = metric_at(base_payload, slice_name, metric)
            row[f"cadence_{label}_{metric}"] = cadence_value
            row[f"base_{label}_{metric}"] = base_value
            row[f"delta_{label}_{metric}"] = safe_delta(cadence_value, base_value)
    rows.append(row)

pair_frame = pd.DataFrame(rows)
if not pair_frame.empty:
    pair_frame["reduced_test_2022_trades"] = pair_frame["delta_test_2022_trades"] < 0
    pair_frame["reduced_test_2022_turnover"] = pair_frame["delta_test_2022_turnover"] < 0
    pair_frame = pair_frame.sort_values(
        ["reduced_test_2022_trades", "delta_test_2022_trades", "delta_test_2022_turnover", "delta_test_2022_return"],
        ascending=[False, True, True, False],
    ).reset_index(drop=True)

pair_path = CARD / "cadence_pair_metrics.csv"
pair_frame.to_csv(pair_path, index=False)
print(f"wrote {pair_path.relative_to(ROOT)} rows={len(pair_frame)}")
display(pair_frame.head(10))

wrote runtime/model_selection/card/quantbench_regime_cadence_screen_v1/cadence_pair_metrics.csv rows=30


,cadence_profile,base_profile,model_name,paper_family,risk_overlay,trade_cadence,cadence_proxy_return,base_proxy_return,delta_proxy_return,cadence_proxy_max_dd,...,base_latest_2025_2026_turnover,delta_latest_2025_2026_turnover,cadence_latest_2025_2026_trades,base_latest_2025_2026_trades,delta_latest_2025_2026_trades,cadence_latest_2025_2026_cadence_blocked_days,base_latest_2025_2026_cadence_blocked_days,delta_latest_2025_2026_cadence_blocked_days,reduced_test_2022_trades,reduced_test_2022_turnover
0,quantbench_regime_cadence_screen_v1_ridge_mse_...,quantbench_regime_cadence_screen_v1_ridge_mse,ridge_mse,linear_regression,none,regime_cadence_fast_bull_slow_stress_v1,553.395277,860.038237,-306.642961,68.803486,...,16.706414,-5.296793,112.0,343.0,-231.0,251.0,NaN,NaN,True,True
1,quantbench_regime_cadence_screen_v1_tcn_mse_re...,quantbench_regime_cadence_screen_v1_tcn_mse,tcn_mse,sequence_convolutional,none,regime_cadence_fast_bull_slow_stress_v1,908.894721,689.381819,219.512902,67.634887,...,22.005539,-7.827405,115.0,343.0,-228.0,228.0,NaN,NaN,True,True
2,quantbench_regime_cadence_screen_v1_logistic_c...,quantbench_regime_cadence_screen_v1_logistic_clf,logistic_clf,linear_classifier,none,regime_cadence_fast_bull_slow_stress_v1,564.508685,733.115212,-168.606526,66.498451,...,19.195044,-7.081050,121.0,343.0,-222.0,242.0,NaN,NaN,True,True
3,quantbench_regime_cadence_screen_v1_market_gra...,quantbench_regime_cadence_screen_v1_market_gra...,market_graph_mse,relational_graph,none,regime_cadence_fast_bull_slow_stress_v1,681.098290,1095.454850,-414.356559,70.279878,...,18.000875,-6.992420,115.0,343.0,-228.0,248.0,NaN,NaN,True,True
4,quantbench_regime_cadence_screen_v1_sequence_l...,quantbench_regime_cadence_screen_v1_sequence_l...,sequence_lag_ridge_mse,sequence_temporal,none,regime_cadence_fast_bull_slow_stress_v1,302.041251,455.445641,-153.404389,75.074529,...,16.466181,-5.418076,119.0,343.0,-224.0,244.0,NaN,NaN,True,True
5,quantbench_regime_cadence_screen_v1_random_for...,quantbench_regime_cadence_screen_v1_random_for...,random_forest_clf,tree,none,regime_cadence_fast_bull_slow_stress_v1,705.811048,593.855044,111.956004,61.178866,...,23.059767,-8.932945,129.0,343.0,-214.0,217.0,NaN,NaN,True,True
6,quantbench_regime_cadence_screen_v1_market_gra...,quantbench_regime_cadence_screen_v1_market_gra...,market_graph_mse,relational_graph,macro_budget_defensive_v1,regime_cadence_fast_bull_slow_stress_v1,107.889269,86.687275,21.201994,26.225608,...,10.110962,-1.342507,46.0,92.0,-46.0,217.0,NaN,NaN,True,True
7,quantbench_regime_cadence_screen_v1_ridge_mse_...,quantbench_regime_cadence_screen_v1_ridge_mse_...,ridge_mse,linear_regression,macro_budget_defensive_v1,regime_cadence_fast_bull_slow_stress_v1,101.314656,86.965338,14.349318,25.860132,...,9.439592,-1.072303,41.0,76.0,-35.0,231.0,NaN,NaN,True,True
8,quantbench_regime_cadence_screen_v1_ridge_mse_...,quantbench_regime_cadence_screen_v1_ridge_mse_...,ridge_mse,linear_regression,macro_budget_bull_guard_v1,regime_cadence_fast_bull_slow_stress_v1,-2.265267,0.125336,-2.390603,25.524875,...,3.713469,-0.040000,14.0,17.0,-3.0,180.0,NaN,NaN,True,True
9,quantbench_regime_cadence_screen_v1_market_gra...,quantbench_regime_cadence_screen_v1_market_gra...,market_graph_mse,relational_graph,macro_budget_bull_guard_v1,regime_cadence_fast_bull_slow_stress_v1,5.970695,12.850541,-6.879846,25.355887,...,3.711137,-0.040000,14.0,17.0,-3.0,180.0,NaN,NaN,True,True


## Diagnostic Summary

The summary sidecar is intentionally conservative: this screen can show that regime cadence is measurable in the simplified supervised path, but it cannot promote a model or justify removing the existing RL stack by itself.

In [5]:
screen_path = CARD / "supervised_candidate_screen.csv"
screen_frame = pd.read_csv(screen_path)

if pair_frame.empty:
    best_pair: dict[str, Any] | None = None
else:
    best_pair = pair_frame.iloc[0].replace({np.nan: None}).to_dict()

eligible_profiles = screen_frame.loc[screen_frame["eligible"].astype(bool), "profile"].tolist() if not screen_frame.empty else []
turnover_reduction_rows = int(pair_frame.get("reduced_test_2022_turnover", pd.Series(dtype=bool)).sum())
trade_reduction_rows = int(pair_frame.get("reduced_test_2022_trades", pd.Series(dtype=bool)).sum())

summary = {
    "experiment": "quantbench_regime_cadence_screen_v1",
    "decision": "cadence_screen_recorded_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "quantbench_paper_soxl_v2 supervised no-cadence rows",
    "role": "diagnose",
    "changed_layer": ["execution", "model_selection"],
    "target_windows": windows,
    "accepted_tradeoff": "lower stress/range trading frequency or turnover without promotion from a single screen",
    "external_reference": {
        "quantbench": report["paper"],
        "kaggle_jane_street_lessons": [
            "time-ordered validation before promotion",
            "leakage avoidance across train/test boundaries",
            "local slice diagnostics over single leaderboard-like proxy metrics",
        ],
        "kaggle_urls": [
            "https://www.kaggle.com/competitions/jane-street-market-prediction",
            "https://www.kaggle.com/competitions/jane-street-real-time-market-data-forecasting",
        ],
    },
    "data": DATA_CONTEXT,
    "model_names": list(model_names),
    "risk_overlays": [config.name for config in risk_configs],
    "trade_cadences": [config.name for config in cadence_configs],
    "screen_records": int(report["screen"]["records"]),
    "eligible_profiles": eligible_profiles,
    "cadence_pair_rows": int(len(pair_frame)),
    "test_2022_trade_reduction_rows": trade_reduction_rows,
    "test_2022_turnover_reduction_rows": turnover_reduction_rows,
    "best_cadence_pair": best_pair,
    "sidecars": {
        "supervised_summary_json": str((CARD / "supervised_summary.json").relative_to(ROOT)),
        "supervised_candidate_screen_csv": str(screen_path.relative_to(ROOT)),
        "supervised_payloads_json": str(payloads_path.relative_to(ROOT)),
        "cadence_pair_metrics_csv": str(pair_path.relative_to(ROOT)),
    },
    "notes": [
        "This is a non-RL qlib DatasetH-backed supervised screen.",
        "This run uses existing repo SOXL factor packs; it does not yet implement a pure qlib Alpha158 feature replacement.",
        "This run proves the dynamic trade-frequency layer is screenable in supervised backtests, not that it is promotion-ready.",
    ],
    "next_action": (
        "Use this cadence evidence to decide whether the broader RL-removal branch should preserve only supervised "
        "model-selection plus execution-cadence interfaces before adding an Alpha158-compatible feature pack."
    ),
}

summary_path = CARD / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True, allow_nan=True) + "\n", encoding="utf-8")
print(json.dumps({
    "summary": str(summary_path.relative_to(ROOT)),
    "eligible_profiles": len(eligible_profiles),
    "cadence_pair_rows": len(pair_frame),
    "test_2022_trade_reduction_rows": trade_reduction_rows,
    "test_2022_turnover_reduction_rows": turnover_reduction_rows,
}, indent=2, sort_keys=True))

{
  "cadence_pair_rows": 30,
  "eligible_profiles": 0,
  "summary": "runtime/model_selection/card/quantbench_regime_cadence_screen_v1/summary.json",
  "test_2022_trade_reduction_rows": 30,
  "test_2022_turnover_reduction_rows": 30
}


In [6]:
from IPython.display import Markdown, display

best = summary.get("best_cadence_pair") or {}
best_profile = best.get("cadence_profile", "none")
best_base = best.get("base_profile", "none")
best_trade_delta = best.get("delta_test_2022_trades")
best_turnover_delta = best.get("delta_test_2022_turnover")
best_return_delta = best.get("delta_test_2022_return")

conclusion = f"""
## Conclusion

`quantbench_regime_cadence_screen_v1` ran a focused non-RL supervised screen over {len(model_names)} model specs, {len(risk_configs)} risk overlays, and {len(cadence_configs)} regime-cadence overlay using the repo qlib DatasetH feature surface through `{model_test_end}`.

Key metrics: `{report['screen']['records']}` candidate rows, `{len(pair_frame)}` matched cadence pairs, `{trade_reduction_rows}` pairs reduced `test_2022` trades, `{turnover_reduction_rows}` pairs reduced `test_2022` turnover, and `{len(eligible_profiles)}` rows passed the existing gate.

Best diagnostic pair by the cadence-reduction sort: `{best_profile}` versus `{best_base}` with `delta_test_2022_trades={best_trade_delta}`, `delta_test_2022_turnover={best_turnover_delta}`, and `delta_test_2022_return={best_return_delta}`.

Decision: record the cadence screen with no promotion. The dynamic trading-frequency layer is now measurable in the simplified supervised path, but this run does not remove the existing RL stack and does not yet satisfy the separate true qlib Alpha158 replacement requirement.

Artifacts: `supervised_candidate_screen.csv`, `cadence_pair_metrics.csv`, `supervised_payloads.json`, `supervised_summary.json`, and `summary.json` in `{CARD.relative_to(ROOT)}`.
"""
display(Markdown(conclusion))


## Conclusion

`quantbench_regime_cadence_screen_v1` ran a focused non-RL supervised screen over 6 model specs, 4 risk overlays, and 1 regime-cadence overlay using the repo qlib DatasetH feature surface through `2026-06-15`.

Key metrics: `60` candidate rows, `30` matched cadence pairs, `30` pairs reduced `test_2022` trades, `30` pairs reduced `test_2022` turnover, and `0` rows passed the existing gate.

Best diagnostic pair by the cadence-reduction sort: `quantbench_regime_cadence_screen_v1_ridge_mse_regime_cadence_fast_bull_slow_stress_v1` versus `quantbench_regime_cadence_screen_v1_ridge_mse` with `delta_test_2022_trades=-178.0`, `delta_test_2022_turnover=-8.943426294820718`, and `delta_test_2022_return=-4.848087049900002`.

Decision: record the cadence screen with no promotion. The dynamic trading-frequency layer is now measurable in the simplified supervised path, but this run does not remove the existing RL stack and does not yet satisfy the separate true qlib Alpha158 replacement requirement.

Artifacts: `supervised_candidate_screen.csv`, `cadence_pair_metrics.csv`, `supervised_payloads.json`, `supervised_summary.json`, and `summary.json` in `runtime/model_selection/card/quantbench_regime_cadence_screen_v1`.
